# 7. Comparison Checkpoint: PydanticAI vs ToyAIKit

This notebook extends the ToyAIKit comparison by adding PydanticAI's typed tools and dependency injection.

## Comparison checkpoint

| Approach | What you write | What the framework manages |
| --- | --- | --- |
| Handwritten loop | schemas, history, dispatch, stopping | nothing beyond the model client |
| ToyAIKit | tools, prompts, callbacks | loop, display, history, cost helpers |
| PydanticAI | typed agent, dependencies, tools | schema generation, execution, validation, result handling |

The goal is to see the same agent pattern at three abstraction levels.

## 1. Setup and imports

Install the framework in the project environment before running the import cell:

```bash
uv add pydantic-ai
```

PydanticAI supports multiple model providers. This notebook uses the OpenAI model already used in the project.

In [ ]:
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext

from ingestion import build_index, load_faq_data

load_dotenv()
documents = load_faq_data()
index = build_index(documents)
print(f"Loaded and indexed {len(documents)} documents")

## 2. Create typed dependencies

PydanticAI separates the agent's instructions from runtime dependencies. The search index is application data, so pass it as `deps` rather than hiding it in a global tool function.

In [ ]:
from dataclasses import dataclass


@dataclass
class SearchDependencies:
    index: object


deps = SearchDependencies(index=index)

## 3. Define the agent and typed search tool

The model string selects the provider and model. The `@agent.tool` decorator exposes a typed function to the model and gives it access to the runtime index through `RunContext`.

In [ ]:
agent = Agent(
    "openai:gpt-5.4-mini",
    deps_type=SearchDependencies,
    system_prompt="""
You're a course teaching assistant.
Use the search tool for course questions.
Search again with improved keywords when the first result is incomplete.
Answer only from the FAQ context.
""".strip(),
)


@agent.tool
def search(ctx: RunContext[SearchDependencies], query: str) -> list[dict]:
    """Search the LLM Zoomcamp FAQ for matching course information."""
    return ctx.deps.index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

## 4. Run a typo-recovery question

PydanticAI manages the model/tool loop, but the behavior still depends on the same search quality and instructions. Use `Olama` to compare the result with notebook 05.

In [ ]:
result = agent.run_sync(
    "How do I run Olama locally?",
    deps=deps,
)
result.output

## 5. Inspect output, usage, and messages

The exact result API depends on the installed PydanticAI version, so inspect the result object rather than guessing its fields. Common inspection points include the final output, usage, and messages.

In [ ]:
print("Output:", result.output)
print("Result type:", type(result))
print("Available result attributes:", [name for name in dir(result) if not name.startswith("_")])

if hasattr(result, "usage"):
    print("Usage:", result.usage())
if hasattr(result, "all_messages"):
    print("Message count:", len(result.all_messages()))

## 6. Run a normal course question

Compare a direct enrollment question with the typo-recovery run. Record the number of model/tool steps and the quality of the retrieved evidence.

In [ ]:
enrollment_result = agent.run_sync(
    "I just discovered the course. Can I still join it?",
    deps=deps,
)
enrollment_result.output

## 7. Compare PydanticAI with the handwritten loop and ToyAIKit

PydanticAI removes manual schema dictionaries and loop plumbing through typed tools, dependency injection, and an `Agent` abstraction. The search function and prompt still determine answer quality.

**Study checkpoint:** identify which responsibilities moved into the framework and which remain application code: retrieval policy, data access, instructions, provider configuration, validation, and observability.

## 8. Final checklist

- `uv add pydantic-ai` was run in the project environment.
- The model provider and API key are configured.
- The search index is passed through typed dependencies.
- The tool has a type hint and a useful docstring.
- Typo recovery and a normal course question were compared.
- Usage, messages, errors, and tool calls are observable.
- The framework is evaluated against the simpler handwritten loop before adoption.